<a href="https://colab.research.google.com/github/tendolkarurja/IPD_Finance/blob/main/IPD_Fin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df_2022 = pd.read_csv('/content/drive/MyDrive/IPD_sentiments/economic_times_headlines_2022.csv')
df_2023 = pd.read_csv('/content/drive/MyDrive/IPD_sentiments/economic_times_headlines_2023.csv')
df_2024 = pd.read_csv('/content/drive/MyDrive/IPD_sentiments/economic_times_headlines_2024.csv')
df_2025 = pd.read_csv('/content/drive/MyDrive/IPD_sentiments/economic_times_headlines_2025.csv')


In [ ]:
print(df_2022.info())
print(df_2023.info())
print(df_2024.info())
print(df_2025.info())

In [ ]:
df_news = pd.concat([df_2022, df_2023, df_2024, df_2025], axis = 0)
df_news.reset_index(drop=True)

In [ ]:
noise_keywords = ['/bollywood', '/lifestyle', '/panache', '/entertainment', '/magazines', '/astrology', '/horoscope', '/sports']

In [ ]:
occ_noise = {i:0 for i in noise_keywords}

for keyword in noise_keywords:
    occ_noise[keyword] += df_news['Headline link'].str.contains(keyword).sum()

In [ ]:
occ_noise

In [ ]:
df_news.info()

In [ ]:
news_volume_per_day = df_news.groupby('Date').size()
print(news_volume_per_day)

min_news = news_volume_per_day.min()
print(f"The lowest number of headlines on any day was: {min_news}")

In [ ]:
(df_news['Date'] == '29-12-2024').unique()

In [ ]:
df_news['Date'] = pd.to_datetime(df_news['Date'], dayfirst=True)
full_range = pd.date_range(start='2022-01-01', end='2025-06-11')

present_dates = pd.to_datetime(df_news['Date'].unique())
missing_dates = full_range.difference(present_dates)

print(f"Total missing days: {len(missing_dates)}")
print(missing_dates)

In [ ]:
for_2025 = pd.date_range(start='2025-01-01', end='2025-06-11')

missing_dates = for_2025.difference(present_dates)

print(f"Total missing days in 2025: {len(missing_dates)}")
print(missing_dates)

In [ ]:
for keyword in noise_keywords:
    df_noise = df_news[df_news['Headline link'].str.contains(keyword)]
    print(df_noise.groupby('Date').size())

In [ ]:
df_news.drop('Archive', axis =1, inplace = True)

In [ ]:
df_news

In [ ]:
pattern = '|'.join(i for i in noise_keywords)
df_news_clean = df_news[~df_news['Headline link'].str.contains(pattern, case=False, na=False)].copy()

# 3. Reset index to keep it clean (0 to N)
df_news_clean = df_news_clean.reset_index(drop=True)

print(f"Original Records: {len(df_news)}")
print(f"Records after Noise Removal: {len(df_news_clean)}")
print(f"Removed: {len(df_news) - len(df_news_clean)} rows.")

In [ ]:
df_news_clean['Headline link'].unique()

In [ ]:

# Identify records in the 'india' path
tickers = ['RELIANCE', 'ADANI', 'TATA POWER', 'NTPC', 'ONGC', 'JSW', 'NHPC', 'POWER GRID', 'BPCL', 'GAIL', 'IOCL']
# india_mask = df_news_clean['Headline link'].str.contains('/india', case=False, na=False)
ticker_pattern = '|'.join(i for i in tickers)

df_news_clean['ticker_match'] = df_news_clean['Headline'].str.contains(ticker_pattern, case=False, na=False)
print(df_news_clean['ticker_match'].sum())

In [ ]:
# df_esg_6comp = df_news_clean

In [ ]:
# df_india_clean

In [ ]:
df_ticker_specific = df_news_clean[df_news_clean['Headline'].str.contains(ticker_pattern, case=False, na=False)].copy()

In [ ]:
df_ticker_specific

In [ ]:
esg_policy_keywords = (
'CARBON|NET ZERO|RENEWABLE|EMISSIONS|DECARBONIZATION|GREEN HYDROGEN|BIOFUEL|EV|'
'SOLAR|WIND|HYDRO|CLEAN|POLLUTION|WASTE|SPILL|FLY ASH|EFFLUENT|CLIMATE|'
'BIODIVERSITY|ECOLOGY|SUSTAINABILITY|METHANE|SEQUESTRATION|AMMONIA'
)
df_esg_env = df_ticker_specific[df_ticker_specific['Headline'].str.contains(esg_policy_keywords, case=False, na=False)]

In [ ]:
df_esg_env

In [ ]:
def tag_company(headline):
    headline = headline.lower()
    for company in tickers:
        if company.lower() in headline:
            return company
    return None

df_esg_env['company'] = df_esg_env['Headline'].apply(tag_company)


In [ ]:
df_esg_env

In [ ]:
df_esg_env.company.value_counts()

In [ ]:
noise_phrases = [
    "stocks in news", "sensex", "nifty", "top losers", "top gainers",
    "should you buy", "market news", "closing bell", "opening bell",
    "technical view", "stock pick", "brokerage view"
]

# 3. Apply the Filter
def is_preferred(headline):
    headline_lower = str(headline).lower()

    # RULE A: Must mention at least one target company
    if not any(comp.lower() in headline_lower for comp in tickers):
        return False

    # RULE B: Eliminate "Noise" headlines
    if any(noise in headline_lower for noise in noise_phrases):
        return False

    # RULE C: Keep specific environment related news (The Most Preferred)
    esg_triggers = ["green", "solar", "renewable", "emission"]
    if any(tg in headline_lower for tg in esg_triggers):
        return True

    return False # Keep general company news for now, eliminate the rest

# Create the cleaned dataframe
final = df_esg_env[df_esg_env['Headline'].apply(is_preferred)].copy()

print(f"Original Records: {len(df_esg_env)}")
print(f"Cleaned Records: {len(final)}")

In [ ]:
final

In [ ]:
final.company.value_counts()

In [ ]:
final.to_csv('energy_headlines.csv', index =False)